In [1]:
from skimage import measure, color
import numpy as np
import argparse
import imutils
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

## images

In [ ]:
led_img= "C:\\Users\\arite\\Desktop\\ASM RA\\ASM_Killifish_repo\\d48_65_s1.png"
no_led_img = "/Users/asmlabuser1/Scripts_Ari/no_led.png"

In [ ]:
led = cv2.cvtColor(cv2.imread(led_img), cv2.COLOR_BGR2RGB)
# no_led = cv2.cvtColor(cv2.imread(no_led_img), cv2.COLOR_BGR2RGB)
plt.imshow(led)

In [ ]:
print(led[0][0]) #pixel of normal image
led_LAB = color.rgb2lab(led)
# led_LAB = led
print(led_LAB[0][0])

In [ ]:
pixels = []
for i,y in enumerate(led_LAB):
    for j, x in enumerate(y):
        if x[0] > 35 and x[1] > 25 and -20<x[2]<40:
            pixels.append([i,j])
pixels_np = np.asarray(pixels)
print(pixels_np)

In [ ]:
mask = np.zeros((len(led), len(led[0])))
for [i,j] in pixels:
    mask[i][j] = 255
plt.imshow(mask)

In [ ]:
mask2 = cv2.dilate(mask2, None, iterations=1)
mask2 = cv2.erode(mask, None, iterations=3)

plt.imshow(mask2)

In [ ]:
print(no_led[0][0]) #pixel of normal image
no_led_LAB = color.rgb2lab(no_led)
print(no_led_LAB[0][0])

In [ ]:
pixels_no = []
for i,y in enumerate(no_led_LAB):
    for j, x in enumerate(y):
        if x[0] > 35 and x[1] > 25 and -20<x[2]<40:
            pixels_no.append([i,j])
# pixels_np_no = np.asarray(pixels_no)
# print(pixels_np)

In [ ]:
mask_no = np.zeros((1000,1600))
for [i,j] in pixels_no:
    mask_no[i][j] = 255
plt.imshow(mask_no)

In [ ]:
mask_no2 = cv2.erode(mask_no, None, iterations=3)
mask_no2 = cv2.dilate(mask_no2, None, iterations=8)
plt.imshow(mask_no2)

In [ ]:
led_img= "C:\\Users\\arite\\Desktop\\ASM RA\\ASM_Killifish_repo\\f1_noled.png"

led_LAB = cv2.cvtColor(cv2.imread(led_img), cv2.COLOR_BGR2LAB)
# led_LAB = color.rgb2lab(led)
plt.imshow(led_LAB)
and_mask = np.logical_and(led_LAB[:, :, 0] > 25, led_LAB[:, :, 1] > 25)
and_mask = np.logical_and(and_mask, np.logical_and(led_LAB[:, :, 2] > -20, led_LAB[:, :, 2] < 40))
pixels = np.argwhere(and_mask)

mask_bg = np.zeros((len(led_LAB), len(led_LAB[0])))
mask_bg[pixels[:, 0], pixels[:, 1]] = 255

# mask2 = cv2.dilate(mask_bw, None, iterations=3)
# mask_bg = cv2.erode(mask2, None, iterations=5)

plt.imshow(mask_bg)

In [ ]:
# %%timeit

led_img= "C:\\Users\\arite\\Desktop\\ASM RA\\ASM_Killifish_repo\\f1_led.png"
no_led_img = "C:\\Users\\arite\\Desktop\\ASM RA\\ASM_Killifish_repo\\f1_noled.png"
# led_LAB = cv2.cvtColor(cv2.imread(led_img), cv2.COLOR_BGR2Lab)
# led_LAB = cv2.cvtColor(cv2.imread(led_img), cv2.COLOR_BGR2Lab)

# plt.imshow(led_LAB)
# print(led_LAB[800][500])

# print(led_LAB[500][800])
# led_LAB = color.rgb2lab(led)

# and_mask = np.logical_and(led_LAB[:, :, 0] > 25, led_LAB[:, :, 1] > 25)
# and_mask = np.logical_and(and_mask, np.logical_and(led_LAB[:, :, 2] > -20, led_LAB[:, :, 2] < 40))

# pixels = np.argwhere(and_mask)
# lower_threshold = (25*2.55, 25+128, -20+128)
# upper_threshold = (100*2.55, 128+128, 40+128)
# mask = cv2.inRange(led_LAB, lower_threshold, upper_threshold)
# plt.imshow(mask)
led = create_mask(cv2.imread(led_img))
noled = create_mask(cv2.imread(no_led_img))
plt.imshow(led-noled)
# maskcv =  
# mask_bw = np.zeros((len(led_LAB), len(led_LAB[0])))
# mask_bw[pixels[:, 0], pixels[:, 1]] = 255
# mask_without_bg = mask_bw-mask_bg
# mask2 = cv2.dilate(mask_without_bg, None, iterations=3)
# mask_erode = cv2.erode(mask2, None, iterations=5)
print(np.count_nonzero(mask))
# plt.imshow(mask_erode)
# plt.imshow(mask_bw)

Work on videos + is some sort of verification needed??

## video
this section contains the current final implementation. we use video path to find all video files. LED_times array saves the final result for each video before saving that data in a csv. 

create_mask creates a mask given the image/frame. it converts frame to LAB color format and applies a small gaussian blur to decrease noise, before applying a threshold on L for luminance(brightness), and the other 2 channels for filtering the red color. 

the for loop is the main code. for each video, a progress bar is initiated. a capture object is opened to process each frame using opencv. bg flag tracks if a background mask has been created or not, which is done on the first frame. the small bg if loop stores the background mask in bg_mask variable and then turns flag to false. Next, we go through all the frames wherein a mask for the frame is created, after which we subtract background mask from the current frame's mask. we store all the timestamps at which LED is observed, and append the max and the min of this along with the video name to LED_times. After all the videos are processed, this array is saved to a csv in data/output folder

--vis flag is used to visualise background mask, and current mask while LED is detected. 

--video_path can be used to provide path to a folder containing videos if not in data/videos. 

--output_path can be used to provide output path for csv if not data/output






In [55]:
vid_path=Path("./data/videos/")
# capture = cv2.VideoCapture(vid_path)
vid_list = list(vid_path.glob("*.mp4"))
# video = cv2.VideoCapture("d48_65_S1.mp4")
print(vid_list)
LED_times = [["name", "start time(s)", "end time(s)"]]
print(LED_times)

out_path = Path("./data/output/")
vis = False

[PosixPath('data/videos/d48_35_F1.mp4'), PosixPath('data/videos/d48_23_T3.mp4'), PosixPath('data/videos/d48_65_S1.mp4'), PosixPath('data/videos/d48_61_T1.mp4')]
[['name', 'start time(s)', 'end time(s)']]


In [148]:
def get_timestamps(timestamps, debug = False):
    first_timestamp = None
    last_timestamp = None
    time_difference = 0
    final_timestamps = []
    for i, timestamp in enumerate(timestamps):
        if i == 0:
            first_timestamp = i
        else:
            time_difference += timestamp - timestamps[i - 1]
            print(f"time difference is {time_difference}") if debug else None

            if time_difference > 2500 and time_difference < 6000:
                last_timestamp = i
                print(f"replacing last timestamp to {timestamps[i]}") if debug else None

            elif time_difference >= 6000:
                if last_timestamp>first_timestamp:
                    print(f"replacing first timestamp with {timestamps[i]}, appending {timestamps[first_timestamp], timestamps[last_timestamp]}") if debug else None
                    final_timestamps.append([first_timestamp, last_timestamp])
                first_timestamp = i
                time_difference = 0
    
    if time_difference > 2500 and time_difference <6000 and [first_timestamp, last_timestamp] not in final_timestamps:
        final_timestamps.append([first_timestamp, last_timestamp])
        print(f"appending final timestamps {first_timestamp, last_timestamp}") if debug else None

    return final_timestamps


In [137]:
def create_mask(frame):
  frame_LAB = cv2.cvtColor(cv2.GaussianBlur(frame,(5,5),0), cv2.COLOR_BGR2Lab)
  lower_threshold = (27*2.55, 25+128, -20+128)
  upper_threshold = (100*2.55, 128+128, 40+128)
  mask = cv2.inRange(frame_LAB, lower_threshold, upper_threshold)
  return mask

def process_video(video_path):
  video = cv2.VideoCapture(str(video_path))
  print(video_path.name)
  
  vis = True
  total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
  progress_bar = tqdm(total=total_frames)
  
  time_for_video=[]
  bg = True
  mask_bg = []
  
  # Read until video is completed
  while(video.isOpened()):
    # Capture frame-by-frame
    ret, frame = video.read()

    if bg == True and ret == True:
      mask_bg = create_mask(frame)
      bg = False
      if vis == True: 
        cv2.imshow(f'{video_path.name} bg',mask_bg)
    if ret == True:  
      # Press Q on keyboard to  exit, for visualization
      if cv2.waitKey(25) & 0xFF == ord('q'):
        break
      
      mask_frame = create_mask(frame)
      mask_bw = mask_frame - mask_bg
      # print(time_for_video)
      #find number of non-zero pixels and print timestamp and pixels count if nonzero pixels > 2500
      if np.count_nonzero(mask_bw) > 2500:


        time_for_video.append(video.get(cv2.CAP_PROP_POS_MSEC))
        # print(min(time_for_video)) 
        # print timestamp of video frame
        # print(video.get(cv2.CAP_PROP_POS_MSEC))
        # EXPERIMENT IF ERODE AND DILATE IS NEEDED
        if vis == True:  
          mask_dilate = cv2.dilate(mask_bw, None, iterations=3)
          mask_erode = cv2.erode(mask_dilate, None, iterations=4)
          cv2.imshow(f'{video_path.name} mask',mask_erode)
      progress_bar.update(1)
    # Break the loop
    else: 
      break
    
  LED_times.append([video_path.name, min(time_for_video)/1000, max(time_for_video)/1000])
  # When everything done, release the video capture object, progress bar, and close frames
  
  video.release()
  progress_bar.close()
  cv2.destroyAllWindows()
  return time_for_video
  

# np.savetxt(out_path + "LED_times.csv",LED_times, delimiter=',', fmt = ["%s","%.2f", "%.2f"] )

In [160]:
video.release()
progress_bar.close()
cv2.destroyAllWindows()


In [150]:

timestamps = process_video(Path("/Users/asmlabuser1/Scripts_Ari/data/videos/143-160/problem_vids/156_M.mp4"))


156_M.mp4


100%|██████████| 5400/5400 [03:34<00:00, 25.15it/s]


In [151]:
processed = timestamps[:-2]
print(processed)

processed_timestamps = get_timestamps(processed, debug = True)
print([(processed[i[0]], processed[i[1]]) for i in processed_timestamps])

[11344.666666666666, 11478.133333333333, 11611.6, 11644.966666666667, 11678.333333333334, 11745.066666666668, 11778.433333333334, 11878.533333333333, 11945.266666666666, 12012.0, 12879.533333333335, 12912.9, 12946.266666666668, 13013.0, 13079.733333333334, 13313.3, 13346.666666666668, 13380.033333333333, 13413.400000000001, 13446.766666666666, 13513.5, 13580.233333333334, 13613.6, 13680.333333333334, 13747.066666666668, 13813.800000000001, 13847.166666666668, 13880.533333333335, 13913.9, 13947.266666666668, 13980.633333333333, 14014.000000000002, 14047.366666666667, 14080.733333333335, 14114.1, 14147.466666666669, 14180.833333333334, 14214.2, 14247.566666666668, 14280.933333333334, 14347.666666666668, 14381.033333333335, 14414.400000000001, 14447.766666666668, 14481.133333333333, 14514.5, 14547.866666666667, 14581.233333333334, 14614.6, 14647.966666666667, 14714.7, 14814.8, 14848.166666666668, 14881.533333333333, 14914.900000000001, 14948.266666666666, 14981.633333333335, 15015.0, 1504

In [99]:
print(time_for_video)

[89122.36666666667, 89155.73333333335, 89189.1, 89222.46666666667, 89255.83333333334, 89289.20000000001, 89322.56666666668, 89355.93333333333, 89389.3, 89422.66666666667, 89456.03333333334, 89489.40000000001, 89522.76666666666, 89556.13333333333, 89589.5, 89622.86666666667, 89656.23333333334, 89689.6, 89722.96666666666, 89756.33333333334, 89789.70000000001, 89823.06666666668, 89856.43333333335, 89889.8, 89923.16666666667, 89956.53333333334, 89989.90000000001, 90023.26666666668, 90056.63333333333, 90090.0, 90123.36666666667, 90156.73333333334, 90190.1, 90223.46666666666, 90256.83333333333, 90290.2, 90323.56666666667, 90356.93333333335, 90390.30000000002, 90423.66666666667, 90457.03333333334, 90490.40000000001, 90523.76666666668, 90557.13333333335, 90590.5, 90623.86666666667, 90657.23333333334, 90690.6, 90723.96666666667, 90757.33333333333, 90790.7, 90824.06666666667, 90857.43333333333, 90890.8, 90924.16666666667, 90957.53333333334, 90990.90000000001, 91024.26666666668, 91057.63333333335

In [84]:
print(LED_times)
# print(time_for_video)
LED_times.append([str(video_path.name), "{:.2f}".format(min(time_for_video)/1000), "{:.2f}".format(max(time_for_video)/1000)])

[['name', 'start time(s)', 'end time(s)'], ['d48_23_T3.mp4', '89.12', '94.09'], ['d48_23_T3.mp4', '89.12', '94.09']]


In [85]:
np.savetxt(Path(out_path, "LED_times.csv"),LED_times, delimiter=',', fmt = "%s" )

In [ ]:
!python ./scripts/LED_times.py --debug

In [ ]:
# movement of camera?

# # Load the two images
# image1 = cv2.imread("image1.jpg")
# image2 = cv2.imread("image2.jpg")
video = cv2.VideoCapture(str(video_path))

while (video.isOpened()):
    # Capture frame-by-frame
    ret, frame = video.read()
    if ret == True:
        
        # Convert the images to grayscale
        gray1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)

        # Find the transformation matrix
        matrix, _ = cv2.findTransformECC(gray1, gray2, None, cv2.MOTION_AFFINE)

        # Check if the transformation is non-trivial
        if matrix is None or (matrix[0,2]**2 + matrix[1,2]**2) < 1e-10:
            print("Camera has not moved")
        else:
            print("Camera has moved")

In [11]:
!python scripts/LED_times.py --video_path data/videos/143-160/problem_vids --debug

trigger
problems found in {'Too many LED events': [], 'LED not observed': ['146_M.mp4']}
running reduced threshold analysis for 146_M.mp4
146_M.mp4
  2%|▊                                      | 117/5400 [00:00<00:13, 393.95it/s]
 bg created
  3%|█▏                                     | 157/5400 [00:00<00:33, 157.44it/s]

In [ ]:
# No LED found in ['146_M.mp4', '157_S.mp4', '153_S.mp4', '147_S.mp4', '144_F.mp4'] as none were present kinda vibe
# ones malfunction due to the higher threshold 148_t (not working at 25 also), 148_s, 159_m
# 147_F, 148_F is too jerky --> too many events (147_M moves at 10s)
# improved due to threshold 150_t 156_m 150_m (also code)
# improved due to delayed bg 148_t 148_s 159_m 154_f 156_m
# ?? 156_F - drifts after 10s, but led event is from 0?
# 152_S dunno how it got fixed, no detection to detection even tho increased threshold? maybe the delayed mask helped?
# troubleshoot from main 

In [1]:
# for trying individual videos
import scripts.LED_times as LED_times_py
from pathlib import Path
LED_times = [["name", "start time(s)", "end time(s)", "start frame", "end frame"]]

problem_vids = {'Too many LED events': [], 'LED not observed': []}
debug = True
vis = False
max_LED = 2
video_path = Path("/Users/asmlabuser1/Scripts_Ari/data/videos/143-160/problem_vids/148_S.mp4")
LED_times_py.process_video(video_path, vis, debug, max_LED, LED_times, problem_vids, sensitive= True)


148_S.mp4


  2%|▏         | 130/6300 [00:00<00:13, 455.12it/s]


 bg created


100%|██████████| 6300/6300 [04:00<00:00, 26.23it/s]

[88988.9, 89022.26666666666, 89055.63333333333, 89089.0, 89122.36666666667, 89155.73333333335, 89189.1, 89222.46666666667, 89255.83333333334, 89289.20000000001, 89322.56666666668, 89355.93333333333, 89389.3, 89422.66666666667, 89456.03333333334, 89489.40000000001, 89522.76666666666, 89556.13333333333, 89589.5, 89622.86666666667, 89656.23333333334, 89689.6, 89722.96666666666, 89756.33333333334, 89789.70000000001, 89823.06666666668, 89856.43333333335, 89889.8, 89923.16666666667, 89956.53333333334, 89989.90000000001, 90023.26666666668, 90056.63333333333, 90090.0, 90123.36666666667, 90156.73333333334, 90190.1, 90223.46666666666, 90256.83333333333, 90290.2, 90323.56666666667, 90356.93333333335, 90390.30000000002, 90423.66666666667, 90457.03333333334, 90490.40000000001, 90523.76666666668, 90557.13333333335, 90590.5, 90623.86666666667, 90657.23333333334, 90690.6, 90723.96666666667, 90757.33333333333, 90790.7, 90824.06666666667, 90857.43333333333, 90890.8, 90924.16666666667, 90957.53333333334,

True

### experimenting with dynamic thresholding

Syntax: cv2.calcHist(images, channels, mask, histSize, ranges[, hist[, accumulate]])

Parameters:

images: list of images as numpy arrays. All images must be of the same dtype and same size.

channels: list of the channels used to calculate the histograms.

mask: optional mask (8 bit array) of the same size as the input image.

histSize: histogram sizes in each dimension

ranges: Array of the dims arrays of the histogram bin boundaries in each dimension

hist: Output histogram

accumulate: accumulation flag, enables to compute a single histogram from several sets of arrays.

Return: It returns an array of histogram points of dtype float32.



In [5]:
# %%timeit
def create_mask(frame):
    frame_LAB = cv2.cvtColor(cv2.GaussianBlur(frame,(5,5),0), cv2.COLOR_BGR2Lab)
    lower_threshold = (0, 25+128, -20+128)
    upper_threshold = (100*2.55, 128+128, 40+128)
    mask_color = cv2.inRange(frame_LAB, lower_threshold, upper_threshold)

    # Calculate histogram of grayscale frame
    hist = cv2.calcHist([frame_LAB], [0], mask_color, [256], [0,256])
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(hist)
    threshold = max_loc[1]
    print(max_loc)
    print('threshold', threshold)

    # # Use threshold value to create lower and upper bounds for mask
    # lower_threshold = (35, 25+128, -20+128)
    # upper_threshold = (100*2.55, 128+128, 40+128)
    # mask = cv2.inRange(frame_LAB, lower_threshold, upper_threshold)
    # return mask


In [6]:
# %%timeit
video = cv2.VideoCapture("data/videos/week_12/124-127/125_M.mp4")
ret, frame = video.read()
mask = create_mask(frame)
# plt.imshow(mask, cmap = 'gray')


(0, 69)
threshold 69


 31%|███▏      | 1864/5940 [01:20<02:39, 25.58it/s]

In [ ]:
33.2, 34.9

In [38]:
a = [1,2]
print(a[3]) if a[3] else print("no")

# how to check if index exists in list
a = [1,2]
if len(a) > 3:
    print(a[3])
else:
    

IndexError: list index out of range

In [1]:
%run -i scripts/LED_times.py --video_path data/videos/week_12/124-127/ --debug

 total videos found are 16
 Videos to be processed are ['125_M.mp4', '127_M.mp4', '126_M.mp4', '124_M.mp4', '124_F.mp4', '126_S.mp4', '126_F.mp4', '124_S.mp4', '126_T.mp4', '124_T.mp4', '125_S.mp4', '127_F.mp4', '127_S.mp4', '125_F.mp4', '125_T.mp4', '127_T.mp4']
125_M.mp4


  3%|▎         | 151/5940 [00:00<00:18, 307.06it/s]

threshold is 30.196078431372552

 bg created


100%|██████████| 5940/5940 [03:45<00:00, 26.37it/s]


[]
[]
LED not observed in 125_M.mp4
127_M.mp4


...

[91224.46666666667, 91257.83333333334, 91291.2, 91324.56666666667, 91357.93333333333, 91391.3, 91424.66666666667, 91458.03333333333, 91491.4, 91524.76666666666, 91558.13333333335, 91591.50000000001, 91624.86666666668, 91658.23333333334, 91691.6, 91724.96666666667, 91758.33333333334, 91791.70000000001, 91825.06666666667, 91858.43333333333, 91891.8, 91925.16666666667, 91958.53333333334, 91991.9, 92025.26666666666, 92058.63333333333, 92092.0, 92125.36666666667, 92158.73333333335, 92192.1, 92225.46666666667, 92258.83333333334, 92292.20000000001, 92325.56666666668, 92358.93333333333, 92392.3, 92425.66666666667, 92459.03333333334, 92492.40000000001, 92525.76666666666, 92559.13333333333, 92592.5, 92625.86666666667, 92659.23333333334, 92692.6, 92725.96666666666, 92759.33333333334, 92792.70000000001, 92826.06666666668, 92859.43333333335, 92892.8, 92926.16666666667, 92959.53333333334, 92992.90000000001, 93026.26666666668, 93059.63333333333, 93093.0, 93126.36666666667, 93159.73333333334, 93193.1,

  2%|▏         | 109/5400 [00:00<00:09, 548.54it/s]

threshold is 20.0

 bg created


100%|██████████| 5400/5400 [03:24<00:00, 26.44it/s]

[5038.366666666667, 5071.733333333334, 5105.1, 5138.466666666667, 5171.833333333334, 5205.200000000001, 5238.5666666666675, 5271.933333333334, 5305.3, 5338.666666666667, 5372.033333333334, 5405.400000000001, 5438.766666666667, 5472.133333333334, 5505.500000000001, 5538.866666666668, 6506.5, 6539.866666666667, 6573.233333333334, 6606.6, 6639.966666666667, 6673.333333333334, 6706.700000000001, 6740.0666666666675, 6773.433333333334, 6806.8, 6973.633333333334, 7007.000000000001, 7040.366666666668, 7107.1, 7140.466666666667, 7173.833333333334, 7207.200000000001, 7240.566666666667, 7273.933333333333, 7307.3, 7374.033333333334, 7640.966666666667, 87487.40000000001, 87520.76666666668, 87554.13333333335, 87587.5, 87620.86666666667, 87654.23333333334, 87687.6, 87720.96666666667, 87754.33333333333, 87787.7, 87821.06666666667, 87854.43333333333, 87887.8, 87921.16666666667, 87954.53333333334, 87987.90000000001, 88021.26666666668, 88054.63333333335, 88088.00000000001, 88121.36666666667, 88154.733333

In [ ]:

%run -t scripts/LED_times.py --video_path data/videos/week_12/ --no_hist_thresh --output_path data/output/no_thresh/